In [ ]:
from flax import linen as nn
import jax.numpy as jnp
import jax
from configs.poisson_fno_config import FNO2DConfig


class Poisson2d:


    def __call__(self, model, **kwds)-> jnp.ndarray:
        pass


        config = FNO2DConfig()


        n_layers = config.n_layers
        last_block_name = f'FNOBlock_{n_layers-1}'
        print(last_block_name)

        # The params are stored inside state.params
        last_block_params = state.params[last_block_name]

        print(f"Successfully accessed {last_block_name} parameters!")

        projection_params = state.params['ChannelMLP_1']

        # Unpack the parameters for the first layer of the MLP
        # Note: If scirex uses Conv layers instead of Dense, these might be 'Conv_0' and 'Conv_1'
        W1 = projection_params['dense_0']['kernel']
        b1 = projection_params['dense_0']['bias']

        # Unpack the parameters for the second layer
        W2 = projection_params['dense_1']['kernel']
        b2 = projection_params['dense_1']['bias']

In [ ]:
@jax.jit 
def compute_both_derivatives(W1, W2, b1, b2, x):
    """
    Computes both the first and second derivatives of the projection layer wrt input x.
    x shape: (batch_size, nx, in_channels)
    """

    # 1. The base function for a single point
    dummy_projection_layer = lambda x_point: jnp.dot(nn.gelu(jnp.dot(x_point, W1) + b1), W2) + b2

    # 2. Define the derivative functions (still for a single point)
    jac1_fn = jax.jacfwd(dummy_projection_layer, argnums=0)  # First derivative
    jac2_fn = jax.jacfwd(jac1_fn, argnums=0)                 # Second derivative

    # 3. Create a wrapper that returns BOTH for a single point
    def point_derivatives(x_point):
        return jac1_fn(x_point), jac2_fn(x_point)

    # 4. Vectorize over the spatial grid (nx) and batch
    # vmap is smart enough to handle the tuple output!
    vmap_nx = jax.vmap(point_derivatives, in_axes=0)
    vmap_batch = jax.vmap(vmap_nx, in_axes=0)

    # 5. Execute! This returns a tuple: (batched_jacobian, batched_hessian)
    return vmap_batch(x)

def comput_firstDerv_MLP(W1, W2, b1, b2, x):
    """
    Computes the first derivative of the projection layer wrt input x.
    x shape: (batch_size, nx, in_channels)
    """

    # 1. The base function for a single point
    dummy_projection_layer = lambda x_point: jnp.dot(nn.gelu(jnp.dot(x_point, W1) + b1), W2) + b2

    # 2. Define the derivative function (for a single point)
    jac_fn = jax.jacfwd(dummy_projection_layer, argnums=0)  # First derivative

    # 3. Vectorize over the spatial grid (nx) and batch
    vmap_nx = jax.vmap(jac_fn, in_axes=0)
    vmap_batch = jax.vmap(vmap_nx, in_axes=0)

    # 4. Execute! This returns the batched Jacobian
    return vmap_batch(x)